# Generic 96 kHz four-band filter design

Edit `SOURCE_CONFIG_JSON` in the **one code cell**. Run it to see the per-filter and four-band plots, then copy the JSON between `BEGIN COMPILE-READY JSON` and `END COMPILE-READY JSON` into a local `.json` file. Alternatively set `OUTPUT_JSON_PATH` to an absolute path to write that file directly. The notebook imports NumPy, SciPy and Matplotlib, and does not import background project code. The starting bank is a public bench example only. See `docs/filter_format.md` for filter aliases, 96 kHz conversion and the build command.


In [ ]:
"""Single-cell generic four-stereo-output filter design at 96 kHz."""

import copy
import json
import math
from dataclasses import dataclass
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# 1. Edit this parameter recipe: frequencies are in Hz; it is a
#    generic bench bank, not a speaker-specific crossover. No other Python
#    files or project modules are needed: run this whole notebook cell.
#    COEFF/BIQUAD/FREE entries from another sample rate cannot be converted.
SOURCE_CONFIG_JSON = r'''
{
  "name": "GENERIC_FOUR_BAND_DEMO",
  "sample_rate": 96000,
  "block_size": 512,
  "approved_for_hardware": true,
  "input_headroom_db": -6.0,
  "max_delay_samples": 2048,
  "bands": [
    {
      "id": "LOW",
      "label": "Low output",
      "gain_db": -12.0,
      "polarity": 1,
      "delay_samples": 0,
      "filters": [
        {
          "type": "LR4_LP",
          "freq": 250.0
        }
      ]
    },
    {
      "id": "LOW_MID",
      "label": "Low-mid output",
      "gain_db": -12.0,
      "polarity": 1,
      "delay_samples": 0,
      "filters": [
        {
          "type": "LR4_HP",
          "freq": 250.0
        },
        {
          "type": "LR4_LP",
          "freq": 1500.0
        }
      ]
    },
    {
      "id": "HIGH_MID",
      "label": "High-mid output",
      "gain_db": -12.0,
      "polarity": 1,
      "delay_samples": 0,
      "filters": [
        {
          "type": "LR4_HP",
          "freq": 1500.0
        },
        {
          "type": "LR4_LP",
          "freq": 6000.0
        }
      ]
    },
    {
      "id": "HIGH",
      "label": "High output",
      "gain_db": -12.0,
      "polarity": 1,
      "delay_samples": 0,
      "filters": [
        {
          "type": "LR4_HP",
          "freq": 6000.0
        }
      ]
    }
  ]
}
'''

# 2. Target and graph settings. Optional path saves the compiled JSON;
#    leave it empty to copy the JSON printed between the BEGIN/END markers.
TARGET_SAMPLE_RATE = 96000
TARGET_BLOCK_SIZE = 512
OUTPUT_JSON_PATH = ""
X_MIN_HZ, X_MAX_HZ = 10.0, 30000.0
Y_MIN_DB, Y_MAX_DB = -80.0, 15.0
PLOT_POINTS = 4096
SHOW_INDIVIDUAL_BIQUADS = True
SHOW_ELECTRICAL_SUM = False  # Does not model the loudspeakers or room.

# 3. Everything below is the coefficient calculator and plotting code.
#    Only NumPy, SciPy and Matplotlib are third-party dependencies.
# Filter formulas and aliases are fully defined below.
SUPPORTED_FILTER_TYPES = (
    "COEFF", "BIQUAD", "FREE",
    "LR2_LP", "LP2_LR", "LR2_HP", "HP2_LR",
    "LR4_LP", "LP4_LR", "LR4_HP", "HP4_LR",
    "LP2", "HP2", "PEQ2", "PEAK2", "PEQ4", "PEAK4",
    "BW2_LP", "LP2_BW", "BW2_HP", "HP2_BW",
    "BW4_LP", "LP4_BW", "BW4_HP", "HP4_BW",
    "PK2", "PK4",
    "LOWSHELF2", "LS2", "HIGHSHELF2", "HS2",
    "LOWSHELF4", "LS4", "HIGHSHELF4", "HS4",
)


@dataclass(frozen=True)
class Biquad:
    b0: float
    b1: float
    b2: float
    a1: float
    a2: float
    label: str

    def sos(self) -> np.ndarray:
        return np.array([self.b0, self.b1, self.b2, 1.0, self.a1, self.a2])


def _validate(fs: float, freq: float, q: float) -> None:
    if not (0.0 < freq < fs / 2.0):
        raise ValueError(f"frequency {freq} must be between 0 and Nyquist ({fs/2})")
    if q <= 0.0:
        raise ValueError("Q must be positive")


def _normalise(b0: float, b1: float, b2: float,
               a0: float, a1: float, a2: float, label: str) -> Biquad:
    return Biquad(b0 / a0, b1 / a0, b2 / a0, a1 / a0, a2 / a0, label)


def lowpass(fs: float, freq: float, q: float, label: str = "LP2") -> Biquad:
    _validate(fs, freq, q)
    w0 = 2.0 * math.pi * freq / fs
    c, s = math.cos(w0), math.sin(w0)
    alpha = s / (2.0 * q)
    return _normalise((1-c)/2, 1-c, (1-c)/2, 1+alpha, -2*c, 1-alpha, label)


def highpass(fs: float, freq: float, q: float, label: str = "HP2") -> Biquad:
    _validate(fs, freq, q)
    w0 = 2.0 * math.pi * freq / fs
    c, s = math.cos(w0), math.sin(w0)
    alpha = s / (2.0 * q)
    return _normalise((1+c)/2, -(1+c), (1+c)/2, 1+alpha, -2*c, 1-alpha, label)


def peaking(fs: float, freq: float, q: float, gain_db: float,
            label: str = "PEQ2") -> Biquad:
    _validate(fs, freq, q)
    a = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * math.pi * freq / fs
    c, s = math.cos(w0), math.sin(w0)
    alpha = s / (2.0 * q)
    return _normalise(1+alpha*a, -2*c, 1-alpha*a,
                      1+alpha/a, -2*c, 1-alpha/a, label)


def shelf(fs: float, freq: float, q: float, gain_db: float,
          high: bool, label: str) -> Biquad:
    """RBJ shelf with Q mapped to shelf slope S using alpha formula."""
    _validate(fs, freq, q)
    a = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * math.pi * freq / fs
    c, s = math.cos(w0), math.sin(w0)
    # This Q-form matches CamillaDSP/RBJ-style shelf parameterisation.
    alpha = s / (2.0 * q)
    two_sqrt_a_alpha = 2.0 * math.sqrt(a) * alpha
    if high:
        b0 = a * ((a+1) + (a-1)*c + two_sqrt_a_alpha)
        b1 = -2*a * ((a-1) + (a+1)*c)
        b2 = a * ((a+1) + (a-1)*c - two_sqrt_a_alpha)
        a0 = (a+1) - (a-1)*c + two_sqrt_a_alpha
        a1 = 2 * ((a-1) - (a+1)*c)
        a2 = (a+1) - (a-1)*c - two_sqrt_a_alpha
    else:
        b0 = a * ((a+1) - (a-1)*c + two_sqrt_a_alpha)
        b1 = 2*a * ((a-1) - (a+1)*c)
        b2 = a * ((a+1) - (a-1)*c - two_sqrt_a_alpha)
        a0 = (a+1) + (a-1)*c + two_sqrt_a_alpha
        a1 = -2 * ((a-1) + (a+1)*c)
        a2 = (a+1) + (a-1)*c - two_sqrt_a_alpha
    return _normalise(b0, b1, b2, a0, a1, a2, label)


def design_filter(spec: dict, fs: float) -> list[Biquad]:
    kind = spec["type"].upper()
    f = float(spec.get("freq", 0.0))
    q = float(spec.get("q", 1.0 / math.sqrt(2.0)))
    gain = float(spec.get("gain_db", 0.0))
    if kind in ("COEFF", "BIQUAD", "FREE"):
        values = [float(spec[key]) for key in ("b0", "b1", "b2", "a1", "a2")]
        return [Biquad(*values, str(spec.get("label", kind)))]
    if kind in ("LR2_LP", "LP2_LR"):
        return [lowpass(fs, f, 0.5, kind)]
    if kind in ("LR2_HP", "HP2_LR"):
        return [highpass(fs, f, 0.5, kind)]
    if kind in ("LR4_LP", "LP4_LR"):
        return [lowpass(fs, f, 1/math.sqrt(2), f"{kind}:1"),
                lowpass(fs, f, 1/math.sqrt(2), f"{kind}:2")]
    if kind in ("LR4_HP", "HP4_LR"):
        return [highpass(fs, f, 1/math.sqrt(2), f"{kind}:1"),
                highpass(fs, f, 1/math.sqrt(2), f"{kind}:2")]
    if kind == "LP2":
        return [lowpass(fs, f, q, kind)]
    if kind == "HP2":
        return [highpass(fs, f, q, kind)]
    if kind in ("BW2_LP", "LP2_BW"):
        return [lowpass(fs, f, 1/math.sqrt(2), kind)]
    if kind in ("BW2_HP", "HP2_BW"):
        return [highpass(fs, f, 1/math.sqrt(2), kind)]
    if kind in ("BW4_LP", "LP4_BW"):
        return [lowpass(fs, f, 0.541196100146197, f"{kind}:1"),
                lowpass(fs, f, 1.306562964876377, f"{kind}:2")]
    if kind in ("BW4_HP", "HP4_BW"):
        return [highpass(fs, f, 0.541196100146197, f"{kind}:1"),
                highpass(fs, f, 1.306562964876377, f"{kind}:2")]
    if kind in ("PEQ2", "PEAK2", "PK2"):
        return [peaking(fs, f, q, gain, kind)]
    if kind in ("PEQ4", "PEAK4", "PK4"):
        return [peaking(fs, f, q, gain/2, f"{kind}:1"),
                peaking(fs, f, q, gain/2, f"{kind}:2")]
    if kind in ("LOWSHELF2", "LS2"):
        return [shelf(fs, f, q, gain, False, kind)]
    if kind in ("HIGHSHELF2", "HS2"):
        return [shelf(fs, f, q, gain, True, kind)]
    if kind in ("LOWSHELF4", "LS4"):
        return [shelf(fs, f, q, gain/2, False, f"{kind}:1"),
                shelf(fs, f, q, gain/2, False, f"{kind}:2")]
    if kind in ("HIGHSHELF4", "HS4"):
        return [shelf(fs, f, q, gain/2, True, f"{kind}:1"),
                shelf(fs, f, q, gain/2, True, f"{kind}:2")]
    raise ValueError(f"unsupported filter type: {kind}")


def finite(value, what):
    if isinstance(value, bool):
        raise ValueError(f"{what} must be a finite number")
    try:
        result = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{what} must be a finite number") from exc
    if not math.isfinite(result):
        raise ValueError(f"{what} must be a finite number")
    return result


def positive_integer(value, what):
    number = finite(value, what)
    if number <= 0 or not number.is_integer():
        raise ValueError(f"{what} must be a positive integer")
    return int(number)


def nonnegative_integer(value, what):
    number = finite(value, what)
    if number < 0 or not number.is_integer():
        raise ValueError(f"{what} must be a nonnegative integer")
    return int(number)


def validate_section(section, band_id, name):
    # The firmware uses float32. Evaluate the plots using those same rounded
    # coefficients; reject a section if rounding makes its poles unstable.
    values = tuple(float(np.float32(v)) for v in
                   (section.b0, section.b1, section.b2,
                    section.a1, section.a2))
    if not all(math.isfinite(v) for v in values):
        raise ValueError(f"{band_id} {name}: non-finite float32 coefficient")
    quantized = Biquad(*values, section.label)
    poles = np.roots([1.0, quantized.a1, quantized.a2])
    if np.any(np.abs(poles) >= 1.0):
        raise ValueError(f"{band_id} {name}: unstable float32 biquad")
    return quantized


def frequency_response(sections: Iterable[Biquad], fs: float,
                       frequencies: np.ndarray) -> np.ndarray:
    response = np.ones(frequencies.shape, dtype=np.complex128)
    for section in sections:
        _, h = signal.freqz(section.sos()[:3], section.sos()[3:],
                            worN=frequencies, fs=fs)
        response *= h
    return response


def response_db(sections: Iterable[Biquad], fs: float, frequencies: np.ndarray,
                gain_db: float = 0.0, floor_db: float = -300.0) -> np.ndarray:
    h = frequency_response(sections, fs, frequencies)
    mag = 20.0 * np.log10(np.maximum(np.abs(h), 10.0 ** (floor_db / 20.0)))
    return mag + gain_db
def transfer(sections, frequencies, fs):
    return frequency_response(sections, fs, frequencies)


def db(h):
    return 20.0 * np.log10(np.maximum(np.abs(h), 1e-15))


source = json.loads(SOURCE_CONFIG_JSON)
source_fs = positive_integer(source.get("sample_rate"), "source sample_rate")
fs = positive_integer(TARGET_SAMPLE_RATE, "TARGET_SAMPLE_RATE")
positive_integer(TARGET_BLOCK_SIZE, "TARGET_BLOCK_SIZE")
if [b.get("id") for b in source.get("bands", [])] != ["LOW", "LOW_MID", "HIGH_MID", "HIGH"]:
    raise ValueError("Source bands must be LOW, LOW_MID, HIGH_MID, HIGH")
if not isinstance(source.get("approved_for_hardware", False), bool):
    raise ValueError("approved_for_hardware must be a boolean")
headroom_db = finite(source.get("input_headroom_db", 0.0), "input_headroom_db")
if ("max_delay_ms" in source) == ("max_delay_samples" in source):
    raise ValueError("Specify exactly one max_delay_ms or max_delay_samples")
if "max_delay_ms" in source:
    max_delay_ms = finite(source["max_delay_ms"], "max_delay_ms")
else:
    max_delay_ms = nonnegative_integer(
        source["max_delay_samples"], "max_delay_samples") * 1000 / source_fs
if max_delay_ms < 0:
    raise ValueError("Maximum delay cannot be negative")
if not (0 < X_MIN_HZ < X_MAX_HZ < fs/2):
    raise ValueError("Graph x limits must satisfy 0 < min < max < Nyquist")
if Y_MIN_DB >= Y_MAX_DB or int(PLOT_POINTS) < 32:
    raise ValueError("Invalid graph y limits or PLOT_POINTS")

compiled_config = copy.deepcopy(source)
compiled_config["sample_rate"] = fs
compiled_config["block_size"] = TARGET_BLOCK_SIZE
compiled_config["name"] = str(source.get("name", "GENERIC_FOUR_BAND")) + f"_{fs//1000}K_COEFF"
compiled_config.pop("max_delay_ms", None)
compiled_config["max_delay_samples"] = math.ceil(max_delay_ms * fs / 1000)
compiled_config["plot"] = {
    "x_min_hz": X_MIN_HZ, "x_max_hz": X_MAX_HZ,
    "y_min_db": Y_MIN_DB, "y_max_db": Y_MAX_DB, "points": PLOT_POINTS,
}

stages_by_band = {}
sections_by_band = {}
delay_samples_by_band = {}
for old_band, new_band in zip(source["bands"], compiled_config["bands"]):
    band_id = old_band["id"]
    gain_db = finite(old_band.get("gain_db", 0.0), f"{band_id}.gain_db")
    polarity = old_band.get("polarity", 1)
    if polarity not in (-1, 1) or isinstance(polarity, bool):
        raise ValueError(f"{band_id}: polarity must be +1 or -1")
    if "delay_ms" in old_band and "delay_samples" in old_band:
        raise ValueError(f"{band_id}: use delay_ms OR delay_samples")
    if "delay_samples" in old_band:
        if "max_delay_ms" in source:
            raise ValueError(f"{band_id}: delay_samples conflicts with max_delay_ms")
        delay_ms = nonnegative_integer(
            old_band["delay_samples"], f"{band_id}.delay_samples") * 1000/source_fs
    else:
        if "delay_ms" in old_band and "max_delay_samples" in source:
            raise ValueError(f"{band_id}: delay_ms conflicts with max_delay_samples")
        delay_ms = finite(old_band.get("delay_ms", 0), f"{band_id}.delay_ms")
    if not 0 <= delay_ms <= max_delay_ms:
        raise ValueError(f"{band_id}: delay exceeds max_delay")
    new_band.pop("delay_ms", None)
    new_band["gain_db"] = gain_db
    new_band["polarity"] = int(polarity)
    delay_samples_by_band[band_id] = round(delay_ms * fs / 1000)
    new_band["delay_samples"] = delay_samples_by_band[band_id]
    if delay_samples_by_band[band_id] > compiled_config["max_delay_samples"]:
        raise ValueError(f"{band_id}: resampled delay exceeds max_delay_samples")
    new_band["filters"] = []
    all_sections = []
    stages = []
    for index, spec in enumerate(old_band.get("filters", []), start=1):
        name = str(spec.get("type", "?"))
        params = f" @ {finite(spec['freq'], 'freq'):g} Hz" if "freq" in spec else ""
        label = str(spec.get("label", f"{name}{params}"))
        if source_fs != fs and str(spec.get("type", "")).upper() in ("COEFF", "BIQUAD", "FREE"):
            raise ValueError(
                "COEFF/BIQUAD/FREE values cannot be reused at a different sample rate. "
                "Paste the original freq/Q/gain recipe instead.")
        sections = [validate_section(v, band_id, label)
                    for v in design_filter(spec, fs)]
        stages.append((label, sections))
        all_sections.extend(sections)
        for part, sec in enumerate(sections, start=1):
            coeff = {key: getattr(sec, key)
                     for key in ("b0", "b1", "b2", "a1", "a2")}
            new_band["filters"].append({
                "type": "COEFF", "label": f"{index}. {label} [{part}/{len(sections)}]",
                **coeff,
            })
    stages_by_band[band_id] = stages
    sections_by_band[band_id] = all_sections

frequencies = np.geomspace(X_MIN_HZ, X_MAX_HZ, PLOT_POINTS)
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
for ax, band in zip(axes.flat, compiled_config["bands"]):
    band_id = band["id"]
    for label, stage_sections in stages_by_band[band_id]:
        (line,) = ax.plot(frequencies, db(transfer(stage_sections, frequencies, fs)),
                          linewidth=1.3, label=label)
        if SHOW_INDIVIDUAL_BIQUADS and len(stage_sections) > 1:
            for sec in stage_sections:
                ax.plot(frequencies, db(transfer([sec], frequencies, fs)),
                        ":", color=line.get_color(), alpha=0.5, linewidth=0.9)
    band_gain = 10.0 ** (band["gain_db"] / 20.0)
    h_band = transfer(sections_by_band[band_id], frequencies, fs) * band_gain
    ax.plot(frequencies, db(h_band), color="black", linewidth=2.3,
            label="Cumulative + band gain")
    ax.set(title=f"{band_id} — {band.get('label', band_id)}",
           xlim=(X_MIN_HZ, X_MAX_HZ), ylim=(Y_MIN_DB, Y_MAX_DB),
           xlabel="Frequency (Hz)", ylabel="Magnitude (dB)")
    ax.set_xscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8, loc="best")
fig.suptitle("Individual filter stages and cumulative transfer functions (96 kHz)")
plt.show()

fig, (mag_ax, phase_ax) = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                                      constrained_layout=True)
electrical_sum = np.zeros(len(frequencies), dtype=np.complex128)
for band in compiled_config["bands"]:
    band_id = band["id"]
    gain = 10.0 ** (band["gain_db"] / 20.0) * band["polarity"]
    h = transfer(sections_by_band[band_id], frequencies, fs) * gain
    h *= np.exp(-2j*np.pi*frequencies*delay_samples_by_band[band_id]/fs)
    electrical_sum += h
    (line,) = mag_ax.plot(frequencies, db(h), linewidth=1.8, label=band_id)
    # Phase is hard to interpret where a branch is attenuated by >70 dB.
    shown_phase = np.where(db(h) > -70, np.angle(h, deg=True), np.nan)
    phase_ax.plot(frequencies, shown_phase, color=line.get_color(),
                  linewidth=1.2, label=band_id)
if SHOW_ELECTRICAL_SUM:
    mag_ax.plot(frequencies, db(electrical_sum), "k--", linewidth=1.4,
                label="Electrical sum (no speaker acoustics)")
mag_ax.set(ylabel="Magnitude (dB)", ylim=(Y_MIN_DB, Y_MAX_DB),
           title="Four stereo-band transfer functions (includes band gain)")
phase_ax.set(xlabel="Frequency (Hz)", ylabel="Wrapped phase (degrees)",
             ylim=(-180, 180), xlim=(X_MIN_HZ, X_MAX_HZ),
             title="Phase at outputs (includes polarity and sample delay)")
phase_ax.set_xscale("log")
for ax in (mag_ax, phase_ax):
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(loc="best")
plt.show()

# Headroom estimate covers the audible band AND the rest of 0..Nyquist.
estimate_freq = np.geomspace(5.0, fs*0.499, 16384)
print(f"Source: {source.get('name', 'UNNAMED')} ({source_fs} Hz)")
print(f"Target: {compiled_config['name']} ({fs} Hz), {TARGET_BLOCK_SIZE} frames/block")
print("Stereo L/R share each band's coefficients; count is per channel.")
for band in compiled_config["bands"]:
    band_id = band["id"]
    peak = max(db(transfer(sections_by_band[band_id], estimate_freq, fs)))
    peak += band["gain_db"] + headroom_db
    print(f"{band_id}: {len(sections_by_band[band_id])} biquads, "
          f"{delay_samples_by_band[band_id]} delay samples, "
          f"peak single-tone output {peak:.2f} dBFS")
print("Peak estimate is for a steady single tone, not a bound on music transients.")
print("\nBEGIN COMPILE-READY JSON (copy from { through final })")
compile_json = json.dumps(compiled_config, indent=2, ensure_ascii=False,
                          allow_nan=False)
print(compile_json)
print("END COMPILE-READY JSON")

if OUTPUT_JSON_PATH:
    from pathlib import Path
    Path(OUTPUT_JSON_PATH).write_text(compile_json + "\n", encoding="utf-8")
    print(f"Saved compile-ready JSON to {OUTPUT_JSON_PATH}")
